# iGaming Player Churn: A CRISP-DM Walkthrough

This notebook follows the six CRISP-DM phases: **Business Understanding, Data Understanding, Data Preparation, Modeling, Evaluation, and Deployment**. We use player activity data to build a churn-risk model, then explore how its scores might support a retention campaign.

## 1. Business Understanding

### The business question

Acquiring a new player can be expensive. One industry estimate put gambling marketing spend at [$3.9 billion](https://www.5wpr.com/new/the-gambling-industry-spent-3-9-billion-on-marketing-last-year-here-is-what-the-data-shows/), and [iGaming Business](https://igamingbusiness.com/marketing-affiliates/dot-player-churn-a-real-problem/) has also described churn as a persistent challenge. The practical question is: **can we spot signs of disengagement early enough to consider a helpful, responsible intervention?**

Unlike a subscription customer, a player usually does not announce that they are leaving. We define a working churn label as no cash-game, tournament, or deposit activity during the 45 days after a 30-day observation period. The clock begins with each player's first recorded activity, and only players with complete follow-up are included.

We will compare candidate models using stratified cross-validation, evaluate the selected model on a held-out test set, and pay particular attention to precision-recall performance as well as ROC-AUC. Finally, we will estimate campaign economics across thresholds using explicit assumptions for player value, offer cost, and campaign success.

These calculations can help prioritize a pilot; they cannot show that a campaign causes retention. Any real deployment should be tested carefully and include responsible-gambling safeguards.

### Project scope

The supplied data covers demographics, cash games, tournaments, deposits, and withdrawals. Cash games, tournaments, and deposits contribute to the churn label; the feature set focuses on cash-game and deposit behaviour. Withdrawals are profiled but are not model inputs in this version.


## 2. Data Understanding

Before preparing features, we take a practical look at the source files: how many records and players they contain, what dates they cover, whether transaction records connect to known player profiles, and which important fields are missing. These checks help us understand what conclusions the data can support.


In [ ]:
# Import the project helpers and define paths relative to the repository root.
from pathlib import Path
import sys

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Could not find the project src/ folder from the notebook directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import RAW_TABLE_FILES, initialize_clean_views, register_raw_views
from src.feature_engineering import (
    create_player_feature_matrix,
    create_player_temporal_target,
    export_prepared_dataset,
)
from src.model import (
    DEFAULT_PARAMETER_GRID,
    evaluate_model,
    save_model_payload,
    train_and_tune_model,
)


In [ ]:
# DuckDB keeps source-table profiling and feature creation in one local session.
con = duckdb.connect(database=':memory:')


In [ ]:
# Keep the raw input paths in one place; the files themselves stay outside version control.
DATA_DIR = PROJECT_ROOT / "data" / "raw"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
data = {name: DATA_DIR / filename for name, filename in RAW_TABLE_FILES.items()}
data


In [ ]:
# Register each source CSV as a DuckDB view for the data-understanding checks.
data = register_raw_views(con, DATA_DIR)
print("All five source CSV views are registered.")


In [ ]:
# See all registered views
con.execute("SHOW TABLES").fetchdf()

In [ ]:
# Check shape,number of users, and min/max dates for each table
audit_query = """
SELECT 
    'Demographics' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT UserID) AS unique_users,
    NULL AS min_date,
    NULL AS max_date
FROM Demographics

UNION ALL

SELECT 
    'CashGames',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(Date AS TIMESTAMP)),
    MAX(CAST(Date AS TIMESTAMP))
FROM CashGames

UNION ALL

SELECT 
    'Tournaments',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(Date AS TIMESTAMP)),
    MAX(CAST(Date AS TIMESTAMP))
FROM Tournaments

UNION ALL

SELECT 
    'Deposits',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(ProcessDate AS TIMESTAMP)),
    MAX(CAST(ProcessDate AS TIMESTAMP))
FROM Deposits

UNION ALL

SELECT 
    'Withdrawals',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(ProcessDate AS TIMESTAMP)),
    MAX(CAST(ProcessDate AS TIMESTAMP))
FROM Withdrawals;
"""

# Run the query and fetch into a Pandas dataframe
audit_results = con.execute(audit_query).fetchdf()
audit_results

In [ ]:
# Are there transacting users missing from Demographics? If so, how many?
orphaned_users_query = """
-- Check for transaction users missing from Demographics
SELECT 'Orphan CashGame Users' AS issue_type, COUNT(DISTINCT UserID) AS missing_count
FROM CashGames 
WHERE UserID NOT IN (SELECT UserID FROM Demographics)

UNION ALL

SELECT 'Orphan Deposit Users', COUNT(DISTINCT UserID)
FROM Deposits 
WHERE UserID NOT IN (SELECT UserID FROM Demographics)

UNION ALL

SELECT 'Orphan Withdrawal Users', COUNT(DISTINCT UserID)
FROM Withdrawals 
WHERE UserID NOT IN (SELECT UserID FROM Demographics)

UNION ALL

SELECT 'Orphan Tournament Users', COUNT(DISTINCT UserID)
FROM Tournaments 
WHERE UserID NOT IN (SELECT UserID FROM Demographics);
"""
orphaned_users_results = con.execute(orphaned_users_query).fetchdf()
orphaned_users_results

In [ ]:
# Check for missing demographics data
missing_demographics_query = """
SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(UserID) AS missing_user_ids,
    COUNT(*) - COUNT(SystemAgeAsOfReg) AS missing_reg_dates,
    COUNT(*) - COUNT(CountryID) AS missing_countries,
    COUNT(*) - COUNT(Gender) AS missing_genders
FROM Demographics;
"""

missing_demographics_results = con.execute(missing_demographics_query).fetchdf()
missing_demographics_results

In [ ]:

# Check for missing transaction data in CashGames
missing_transaction_data_query = """
SELECT 
    'CashGames' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(UserID) AS missing_user_ids,
    COUNT(*) - COUNT(Date) AS missing_dates,
    COUNT(*) - COUNT(Windows) AS missing_Windows,
    COUNT(*) - COUNT(StakesC) AS missing_StakesC,
    COUNT(*) - COUNT(WinningsC) AS missing_WinningsC
FROM CashGames
"""
missing_transaction_data = con.execute(missing_transaction_data_query).fetchdf()
missing_transaction_data

In [ ]:
# Check for missing transaction data in Tournaments
missing_tournament_values = """
SELECT
    'tournament' as table_name,
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT("UserID") AS missingID,
    COUNT(*) - COUNT("Date") AS missingDate,
    COUNT(*) - COUNT("Trnmnts") AS missingTornaments,
    COUNT (*) - COUNT("StakesT") AS missingStakes,
    COUNT (*) - COUNT("WinningsT") as missingWinnings

FROM tournaments

"""

missing_transaction_data = con.execute(missing_tournament_values).fetch_df()
missing_transaction_data

In [ ]:
# Check for missing deposit fields; one row represents one deposit transaction.
missing_Deposit_query = """
SELECT 'Deposits' AS table_name,
       COUNT(*) AS total_rows,
       COUNT(*) - COUNT(UserID) AS missing_user_ids,
       COUNT(*) - COUNT(DepositID) AS missing_deposit_ids,
       COUNT(*) - COUNT(SummaryDate) AS missing_summary_dates,
       COUNT(*) - COUNT(ProcessDate) AS missing_process_dates,
       COUNT(*) - COUNT(ProcessTime) AS missing_process_times,
       COUNT(*) - COUNT(PayMeth) AS missing_payment_methods,
       COUNT(*) - COUNT(PayMethCat) AS missing_payment_categories,
       COUNT(*) - COUNT(CardType) AS missing_card_types,
       COUNT(*) - COUNT(Amount) AS missing_amounts,
       COUNT(*) - COUNT(Status) AS missing_statuses
FROM Deposits;
"""
null_deposits_query = con.execute(missing_Deposit_query).fetchdf()
null_deposits_query


In [ ]:
# Check for missing withdrawal fields; keep this table profiled even though it is not a model feature.
missing_withdrawals_query = """
SELECT 'Withdrawals' AS table_name,
       COUNT(*) AS total_rows,
       COUNT(*) - COUNT(UserID) AS missing_user_ids,
       COUNT(*) - COUNT(WithdrawalID) AS missing_withdrawal_ids,
       COUNT(*) - COUNT(SummaryDate) AS missing_summary_dates,
       COUNT(*) - COUNT(ProcessDate) AS missing_process_dates,
       COUNT(*) - COUNT(ProcessTime) AS missing_process_times,
       COUNT(*) - COUNT(PayMeth) AS missing_payment_methods,
       COUNT(*) - COUNT(PayMethCat) AS missing_payment_categories,
       COUNT(*) - COUNT(CardType) AS missing_card_types,
       COUNT(*) - COUNT(Amount) AS missing_amounts,
       COUNT(*) - COUNT(Status) AS missing_statuses
FROM Withdrawals;
"""
missing_withdrawals_data = con.execute(missing_withdrawals_query).fetchdf()
missing_withdrawals_data


### What the initial checks tell us

The inventory and integrity checks above establish the scope of the supplied data and surface missing values or records that do not join to a player profile. We keep these checks visible rather than quietly assuming every source is complete.

Date coverage matters especially for churn work. A player can only be labelled reliably when the data includes their full observation and follow-up window. The target-building query below checks for that completeness before assigning labels.


## 3. Data Preparation

Here we standardize the raw columns, create a player-level churn label, and assemble features from the observation period. The key rule is temporal: **features may use information from the first 30 days only; the outcome is measured in the following 45 days**. Keeping the two windows separate prevents future information from leaking into predictors.


### 3.1 Clean and standardize the source tables

The cleaning layer casts identifiers, dates, and financial fields into consistent types. Text categories are normalized, missing descriptive values receive explicit labels, and missing numeric transaction amounts are set to zero for aggregation. Rows without a usable player ID or event date cannot contribute safely to time-based summaries, so they are excluded from the relevant cleaned view.

We leave the original CSV views untouched and build separate `Clean_*` views, making these choices easy to inspect and repeat.


In [ ]:
# Apply shared type casting, normalization, and missing-value rules from src/.
initialize_clean_views(con)
print("Standardized DuckDB views are ready.")


### 3.2 Create the player-relative outcome

Each player's clock starts at their first recorded cash-game, tournament, or deposit event. We observe 30 days, then look for activity in the following 45 days. All three activity types count toward the outcome, and players without complete follow-up are excluded. This makes the label consistent with the business definition and avoids calling an unfinished history a churn event.


In [ ]:
# Create the player-relative 30-day observation and 45-day outcome label.
create_player_temporal_target(con, observation_days=30, target_days=45)
print("Player-relative churn labels are ready.")

df_check = con.execute("""
SELECT churned,
       COUNT(*) AS player_count,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM Player_Temporal_Target
GROUP BY churned
ORDER BY churned;
""").fetchdf()
display(df_check)


### 3.3 Build features from early behaviour

The feature matrix summarizes cash-game and deposit behaviour up to each player's observation cutoff: session count, stakes, winnings, deposits, recency, and two smoothed ratios. A small `+1.0` in each ratio's denominator avoids division by zero.

The target label and feature cutoff now share the same player-relative timeline. Tournament activity contributes to the outcome label, while the current feature set focuses on cash-game and deposit behaviour.


In [ ]:
# Create features from the observation period only; SQL lives in src/feature_engineering.py.
create_player_feature_matrix(con)
print("The leakage-controlled feature view is ready.")
con.execute("SELECT * FROM Player_Feature_Matrix LIMIT 5;").fetchdf()


### 3.4 Check and export the modelling table

Before training, we check the number of eligible players, feature count, missing-value total, and churn balance. The Parquet export lets the modelling phase load the exact prepared table in a compact, reusable format.


In [ ]:
# Export the prepared view and inspect the dataset before training.
df_model = export_prepared_dataset(
    con,
    ARTIFACTS_DIR / "igaming_churn_prepared_dataset.parquet",
)

print("--- PREPARED DATASET CHECK ---")
print(f"Eligible players: {len(df_model):,}")
print(f"Columns (including ID and target): {len(df_model.columns)}")
print(f"Null values across the feature matrix: {df_model.isnull().sum().sum()}")
print("\n--- CHURN LABEL BALANCE ---")
print((df_model["churned"].value_counts(normalize=True).sort_index() * 100).round(2))
df_model.head()


## 4. Modeling

We set aside a stratified 20% holdout before training. The remaining training set is compared with five-fold stratified cross-validation. LightGBM is retained as the candidate model used by the existing analysis; the holdout remains separate for the next phase.


In [ ]:
# Compare baseline models and tune LightGBM using cross-validation on training data.
df_model = pd.read_parquet(ARTIFACTS_DIR / "igaming_churn_prepared_dataset.parquet")
training_results = train_and_tune_model(df_model)

feature_cols = training_results.feature_columns
baseline_model = training_results.baseline_model
tuned_model = training_results.tuned_model
best_model = training_results.selected_model
selected_model_name = training_results.selected_model_name
baseline_cv_pr_auc = training_results.baseline_cv_pr_auc
tuned_cv_pr_auc = training_results.tuned_cv_pr_auc
best_params = training_results.best_parameters
X_test = training_results.X_test
y_test = training_results.y_test
df_cv = training_results.cross_validation_results

print(f"Prepared training rows: {len(df_model) - len(X_test):,}")
print(f"Held-out test rows: {len(X_test):,}")
print("\n--- BASELINE CROSS-VALIDATION ---")
display(df_cv)


### 4.1 Tune LightGBM with cross-validation

The baseline is a useful starting point, but its settings are only one reasonable choice. This exhaustive, bounded grid tests combinations of tree size, learning rate, and regularization using stratified five-fold cross-validation on the training data. The search selects the configuration with the highest **mean average precision (PR-AUC)**, a useful primary metric when churn and retention classes are imbalanced.

The held-out test set is not used to tune parameters. The winner is the best configuration in this search grid—not a guarantee of a globally optimal model.

In [ ]:
# Report the search results returned by the reusable model-training module.
configurations_tested = 1
for values in DEFAULT_PARAMETER_GRID.values():
    configurations_tested *= len(values)

print(f"Configurations tested: {configurations_tested}")
print(f"Baseline LightGBM mean CV PR-AUC: {baseline_cv_pr_auc:.4f}")
print(f"Tuned LightGBM mean CV PR-AUC:    {tuned_cv_pr_auc:.4f}")
print("Best parameters found:")
print(best_params)
print(f"\nSelected using training-fold PR-AUC: {selected_model_name}")


## 5. Evaluation

We now use held-out players once to estimate how the selected model performs on unseen examples. ROC-AUC summarizes ranking across thresholds; PR-AUC and the class-level report help us understand the churn class. The default 0.50 classification threshold is shown for reference, not as an automatic business choice.


In [ ]:
# Evaluate only after cross-validation has selected the model.
baseline_evaluation = evaluate_model(baseline_model, X_test, y_test)
tuned_evaluation = evaluate_model(tuned_model, X_test, y_test)
selected_evaluation = evaluate_model(best_model, X_test, y_test)

test_model_comparison = pd.DataFrame([
    {
        "Model": "Baseline LightGBM",
        "CV PR-AUC": baseline_cv_pr_auc,
        "Holdout PR-AUC": baseline_evaluation["pr_auc"],
        "Holdout ROC-AUC": baseline_evaluation["roc_auc"],
    },
    {
        "Model": "Tuned LightGBM",
        "CV PR-AUC": tuned_cv_pr_auc,
        "Holdout PR-AUC": tuned_evaluation["pr_auc"],
        "Holdout ROC-AUC": tuned_evaluation["roc_auc"],
    },
])
print("--- BASELINE VS TUNED LIGHTGBM ---")
display(test_model_comparison)

# Keep the selected model's holdout scores available for charts and campaign scenarios.
y_pred_probs = selected_evaluation["probabilities"]
y_pred = selected_evaluation["predictions"]
pr_auc = selected_evaluation["pr_auc"]
roc_auc = selected_evaluation["roc_auc"]
print(f"--- HELD-OUT RESULTS: {selected_model_name} ---")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")
print("Confusion matrix (rows = actual, columns = predicted):")
print(selected_evaluation["confusion_matrix"])
print("\nClassification report:")
display(pd.DataFrame(selected_evaluation["classification_report"]).T)


In [ ]:
# Rank predictors using the importance measure exposed by the fitted LightGBM model.
feature_importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importance', y='Feature', data=feature_importances,
    palette='viridis'
)
plt.title('LightGBM Feature Importance (Split Count)', fontsize=14, fontweight='bold')
plt.xlabel('Feature Usage Count in Trees')
plt.ylabel('Engineered Feature')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

display(feature_importances)


## 6. Deployment

The campaign table estimates the cost and possible retained revenue at different risk thresholds. It assumes every flagged player receives the offer, every offer has the same cost, and 25% of correctly flagged churners are retained. This is a planning scenario—not a guaranteed return. Before production use, test the assumptions in a controlled pilot, monitor model performance, and build responsible-gambling protections into campaign design.


In [ ]:
# Estimate campaign economics; the assumptions below are scenario inputs, not measured effects.
import numpy as np
import pandas as pd

# Define business parameters for iGaming retention campaign
AVG_PLAYER_LTV = 150.0  # Estimated Net Revenue per retained active player ($)
RENEWAL_COST = 20.0     # Cost of targeted bonus/free-bet campaign per player ($)

# Get predictions on holdout test set
y_probs = best_model.predict_proba(X_test)[:, 1]

# Calculate campaign financials across probability thresholds
thresholds = np.arange(0.1, 1.0, 0.05)
campaign_results = []

for t in thresholds:
    # Target players flagged as churn risk above threshold t
    flagged = (y_probs >= t)
    
    tp = np.sum((flagged == 1) & (y_test == 1))  # True Churners Correctly Flagged
    fp = np.sum((flagged == 1) & (y_test == 0))  # Retained Players Falsely Flagged
    
    total_campaign_cost = (tp + fp) * RENEWAL_COST
    # Assuming campaign successfully retains 25% of targeted true churners
    retained_revenue = tp * 0.25 * AVG_PLAYER_LTV
    net_roi = retained_revenue - total_campaign_cost
    
    campaign_results.append({
        'Threshold': round(t, 2),
        'Targeted_Players': tp + fp,
        'True_Churners_Saved': int(tp * 0.25),
        'Campaign_Cost ($)': round(total_campaign_cost, 2),
        'Retained_Revenue ($)': round(retained_revenue, 2),
        'Net_ROI ($)': round(net_roi, 2)
    })

df_business = pd.DataFrame(campaign_results)
best_threshold = df_business.loc[df_business['Net_ROI ($)'].idxmax()]

print("--- OPTIMAL RETENTION CAMPAIGN THRESHOLD ---")
print(df_business.to_string(index=False))
print(f"\nOptimal Decision Threshold: {best_threshold['Threshold']}")
print(f"Projected Net ROI: ${best_threshold['Net_ROI ($)']:,}")


### Closing the loop

CRISP-DM is iterative: deployment should feed new evidence back into business understanding, data checks, and model evaluation. A responsible next step is a limited pilot with a pre-agreed success measure, a comparison group, and clear rules for excluding players who should not receive promotional contact.


In [ ]:
# Save the selected model, ordered feature list, and operational threshold.
model_path = save_model_payload(
    ARTIFACTS_DIR / "igaming_churn_lightgbm_v1.pkl",
    best_model,
    feature_cols,
    best_threshold["Threshold"],
)
print(f"Model payload saved to {model_path.relative_to(PROJECT_ROOT)}.")


In [ ]:
# Refresh the portfolio report from this run's measured results and declared assumptions.
top_features = ", ".join(feature_importances["Feature"].head(3).tolist())
report_text = f"""iGAMING PLAYER CHURN — CRISP-DM PROJECT SUMMARY

BUSINESS UNDERSTANDING
The goal is to identify players at elevated risk of becoming inactive early enough
for a responsible retention team to consider an appropriate intervention.

DATA PREPARATION
Eligible players: {len(df_model):,}
Churn rate: {df_model['churned'].mean():.1%}
Observation window: the first 30 days from each player's first recorded activity.
Outcome window: the following 45 days; incomplete 75-day histories are excluded.
Predictors use cash-game and deposit activity from the observation window.

MODELLING AND EVALUATION
Selected model: {selected_model_name}.
Mean training-fold PR-AUC: {max(baseline_cv_pr_auc, tuned_cv_pr_auc):.4f}
Held-out ROC-AUC: {roc_auc:.4f}
Held-out PR-AUC: {pr_auc:.4f}
Leading model features: {top_features}

DEPLOYMENT SCENARIO
Highest estimated net ROI threshold: {best_threshold['Threshold']:.2f}
Estimated holdout net ROI: ${best_threshold['Net_ROI ($)']:,.2f}
Assumptions: $150 estimated net value per retained player, a $20 offer cost,
and a 25% campaign success rate. These are planning assumptions, not causal
findings. Validate any intervention with a controlled pilot and apply
responsible-gambling safeguards before using scores operationally.
"""

report_path = REPORTS_DIR / "iGaming_Churn_CRISP_DM_Final_Report.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report_text, encoding="utf-8")
print(f"Updated report at {report_path.relative_to(PROJECT_ROOT)}.")
